# 01 — Knowledge Base (PDF Ingestion)

Ingests Andrew's curated PDF corpus into ChromaDB.

**Sources**: `knowledge_base/sources.json` (32 regulatory/academic/industry) + `knowledge_base/sources_rs.json` (15 R&S vendor)

**Pool mapping**:
- Categories A–I → `trend` (regulatory, academic, industry)
- Categories J–P → `product` (R&S vendor literature)

**Output**: ChromaDB collection + `data/outputs/kb_state.json`

In [ ]:
import sys, os, json, re
sys.path.insert(0, os.path.abspath(".."))

import fitz  # pymupdf
import chromadb

from src.config import CHROMA_PERSIST_DIR
from src.llm import embed
from src.models.kb import Source, Chunk, SourceType, ContentOrigin
from src.models.common import KBPool, stable_id

PDF_DIR = os.path.abspath("../knowledge_base/pdfs")
SOURCES_GENERAL = os.path.abspath("../knowledge_base/sources.json")
SOURCES_RS = os.path.abspath("../knowledge_base/sources_rs.json")

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
try:
    client.delete_collection("knowledge_base")
    print("Cleared existing collection")
except Exception:
    print("No existing collection to clear")
collection = client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"},
)
print(f"ChromaDB at {CHROMA_PERSIST_DIR} — {collection.count()} docs")

In [ ]:
# ── Category → pool / SourceType mappings ──────────────────────────────────
CATEGORY_POOL = {
    "A_regulator_canonical": KBPool.TREND,
    "B_regional_regulator":  KBPool.TREND,
    "C_us_regulator":        KBPool.TREND,
    "D_eu_strategy":         KBPool.TREND,
    "E_industry_international": KBPool.TREND,
    "F_academic_ai_ml":      KBPool.TREND,
    "G_academic_sensing":    KBPool.TREND,
    "H_security":            KBPool.TREND,
    "I_measurement_framework": KBPool.TREND,
    "J_system_overview":     KBPool.PRODUCT,
    "K_receivers":           KBPool.PRODUCT,
    "L_direction_finders":   KBPool.PRODUCT,
    "M_antennas":            KBPool.PRODUCT,
    "N_software_systems":    KBPool.PRODUCT,
    "P_application_notes":   KBPool.PRODUCT,
}

CATEGORY_TYPE = {
    "A_regulator_canonical":    SourceType.REGULATION,
    "B_regional_regulator":     SourceType.REGULATION,
    "C_us_regulator":           SourceType.REGULATION,
    "D_eu_strategy":            SourceType.REGULATION,
    "E_industry_international": SourceType.TREND_REPORT,
    "F_academic_ai_ml":         SourceType.RESEARCH_PAPER,
    "G_academic_sensing":       SourceType.RESEARCH_PAPER,
    "H_security":               SourceType.TECH_ARTICLE,
    "I_measurement_framework":  SourceType.REGULATION,
    "J_system_overview":        SourceType.TECH_ARTICLE,
    "K_receivers":              SourceType.PRODUCT_PAGE,
    "L_direction_finders":      SourceType.PRODUCT_PAGE,
    "M_antennas":               SourceType.PRODUCT_PAGE,
    "N_software_systems":       SourceType.PRODUCT_PAGE,
    "P_application_notes":      SourceType.DATASHEET,
}


def load_entries(path: str) -> list[dict]:
    with open(path) as f:
        data = json.load(f)
    return data["sources"]


entries_general = load_entries(SOURCES_GENERAL)
entries_rs      = load_entries(SOURCES_RS)
all_entries     = entries_general + entries_rs

print(f"Total entries: {len(all_entries)} ({len(entries_general)} general + {len(entries_rs)} R&S)")

In [ ]:
def extract_chunks(pdf_path: str, max_chunk: int = 1000, min_chunk: int = 80) -> list[str]:
    """Extract and chunk text from a PDF using pymupdf."""
    doc = fitz.open(pdf_path)
    pages_text = []
    for page in doc:
        pages_text.append(page.get_text())
    doc.close()

    full_text = "\n\n".join(pages_text)
    full_text = re.sub(r"\n{3,}", "\n\n", full_text)
    full_text = re.sub(r"[ \t]+", " ", full_text)

    paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]

    chunks, buf = [], ""
    for para in paragraphs:
        if len(para) < 25:  # skip page numbers, lone headers
            continue
        if len(buf) + len(para) + 2 <= max_chunk:
            buf = f"{buf}\n\n{para}" if buf else para
        else:
            if len(buf) >= min_chunk:
                chunks.append(buf)
            buf = para
    if len(buf) >= min_chunk:
        chunks.append(buf)

    return chunks


print("extract_chunks() ready")

In [ ]:
sources: dict[str, Source] = {}
chunks:  dict[str, Chunk]  = {}
skipped = []

BATCH = 64  # embed in batches to stay within API limits

pending_ids, pending_texts, pending_metas, pending_source_ids, pending_pool = [], [], [], [], []

def flush_batch():
    if not pending_texts:
        return
    embeddings = embed(pending_texts)
    collection.add(
        ids=pending_ids,
        embeddings=embeddings,
        documents=pending_texts,
        metadatas=pending_metas,
    )
    pending_ids.clear(); pending_texts.clear()
    pending_metas.clear(); pending_source_ids.clear(); pending_pool.clear()


for entry in all_entries:
    filename = entry.get("filename", "")
    pdf_path = os.path.join(PDF_DIR, filename)

    if not filename or not os.path.exists(pdf_path):
        skipped.append(entry["id"])
        print(f"  [SKIP] {entry['id']} — PDF not found")
        continue

    category = entry.get("category", "")
    pool      = CATEGORY_POOL.get(category, KBPool.TREND)
    stype     = CATEGORY_TYPE.get(category, SourceType.TECH_ARTICLE)

    source_id = entry["id"]
    src = Source(
        id=source_id,
        title=entry["title"],
        url=entry.get("url", ""),
        type=stype,
        pool=pool,
        content_origin=ContentOrigin.FETCHED,
    )
    sources[source_id] = src

    raw_chunks = extract_chunks(pdf_path)

    for text in raw_chunks:
        chunk_id = stable_id(source_id, text[:200])
        chunk = Chunk(
            id=chunk_id,
            source_id=source_id,
            content=text,
            section=category,
            pool=pool,
        )
        chunks[chunk_id] = chunk

        pending_ids.append(chunk_id)
        pending_texts.append(text)
        pending_metas.append({
            "source_id":    source_id,
            "source_title": src.title,
            "pool":         pool.value,
            "section":      category,
            "publisher":    entry.get("publisher", ""),
            "year":         str(entry.get("year", "")),
        })

        if len(pending_ids) >= BATCH:
            flush_batch()

    print(f"  [OK] {source_id:55s} {len(raw_chunks):4d} chunks")

flush_batch()

print(f"\n{'='*60}")
print(f"Sources processed : {len(sources)}")
print(f"Sources skipped   : {len(skipped)} {skipped}")
print(f"Total chunks      : {len(chunks)}")
print(f"ChromaDB docs     : {collection.count()}")

In [ ]:
state = {
    "sources": {k: v.model_dump(mode="json") for k, v in sources.items()},
    "chunks":  {k: v.model_dump(mode="json") for k, v in chunks.items()},
}
os.makedirs("../data/outputs", exist_ok=True)
with open("../data/outputs/kb_state.json", "w") as f:
    json.dump(state, f, indent=2, default=str)

product_src = sum(1 for s in sources.values() if s.pool == KBPool.PRODUCT)
trend_src   = sum(1 for s in sources.values() if s.pool == KBPool.TREND)
product_ch  = sum(1 for c in chunks.values()  if c.pool == KBPool.PRODUCT)
trend_ch    = sum(1 for c in chunks.values()  if c.pool == KBPool.TREND)

print(f"Saved kb_state.json")
print(f"  Sources: {product_src} product + {trend_src} trend = {len(sources)} total")
print(f"  Chunks : {product_ch} product + {trend_ch} trend  = {len(chunks)} total")
print(f"  ChromaDB: {collection.count()} documents")

In [ ]:
from src.rag import retrieve

print("=== 'AI spectrum monitoring autonomous future' ===\n")
for r in retrieve(collection, "AI spectrum monitoring autonomous future", n=5):
    print(f"[{r['source_title'][:55]}]")
    print(f"  {r['content'][:120]}...\n")

print("=== 'wideband receiver direction finding geolocation' ===\n")
for r in retrieve(collection, "wideband receiver direction finding geolocation", n=5):
    print(f"[{r['source_title'][:55]}]")
    print(f"  {r['content'][:120]}...\n")